# MoE-MedIR — SOTA Baselines Notebook

> **Conference**: ICARCV 2026

So sánh 4 SOTA image retrieval methods với MoE-MedIR:

| Method | Venue | Ý tưởng |
|---|---|---|
| **GeM** | TPAMI 2019 | Learnable pooling power p để fuse CLS + PatchMean |
| **DOLG** | ICCV 2021 | Orthogonal fusion: local features bổ sung non-redundant info cho global |
| **CSQ** | CVPR 2020 | Binary hash codes với class hash centers (64-bit) |
| **SuperGlobal** | ICCV 2023 | α-QE re-ranking: aggregate top-K neighbors để sharpen query |

**Yêu cầu**: Chạy `kaggle_train.ipynb` Cell 1–6 trước (features phải có sẵn).
**Setup**: GPU T4/P100 · Internet On

## Cell 1 — Check GPU

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')
else:
    raise RuntimeError('No GPU — Settings → Accelerator → GPU')

## Cell 2 — Install Packages

In [ ]:
%%capture
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'medmnist', 'open_clip_torch', 'timm',
    'pytorch-metric-learning', 'scikit-learn',
    'matplotlib', 'seaborn'], check=True)
print('Done.')

## Cell 3 — Clone & Setup

Clone repo và chọn backbone. **Phải dùng cùng backbone với features đã extract.**

In [ ]:
import os, subprocess

REPO_URL     = 'https://github.com/trong5nhan6/moe_medir.git'
PROJECT_ROOT = '/kaggle/working/moe_medir'

if os.path.exists(PROJECT_ROOT):
    subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, PROJECT_ROOT], check=True)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
subprocess.run(['git', 'log', '--oneline', '-3'])

## Cell 4 — ⚙️ Config

> ⚠️ **Quan trọng**: `FEATURE_MODE` phải là `concat` để GeM và DOLG hoạt động đúng.
> Nếu features đã extract với `cls`, chỉ CSQ và SuperGlobal có thể dùng.

In [ ]:
import re, sys, importlib

# ════════════════════════════════════════════════════════════════
#  ✏️  CHỈNH SỬA TẠI ĐÂY
# ════════════════════════════════════════════════════════════════
BACKBONE     = "clip_vitb32"   # phải khớp với features đã extract
FEATURE_MODE = "concat"        # concat: GeM+DOLG+CSQ   |   cls: chỉ CSQ
EMBED_DIM    = 128
NUM_EXPERTS  = 8               # chỉ dùng nếu chạy SuperGlobal trên MoE
BATCH_SIZE   = 256
EPOCHS       = 50
LR           = 1e-4
# ════════════════════════════════════════════════════════════════

def _patch(content, field, value):
    if isinstance(value, str):
        pattern     = rf'([ \t]+{field}\s*:\s*\w+\s*=\s*)["\'\'].*?["\'\']'
        replacement = rf'\g<1>"{value}"'
    else:
        pattern     = rf'([ \t]+{field}\s*:\s*[\w\[\]]+\s*=\s*)[\d.e+\-]+'
        replacement = rf'\g<1>{value}'
    new, n = re.subn(pattern, replacement, content)
    if n == 0:
        print(f"  [warn] field '{field}' not found")
    return new

with open('config.py', 'r') as f:
    cfg_text = f.read()
for field, val in [('backbone', BACKBONE), ('feature_mode', FEATURE_MODE),
                   ('embed_dim', EMBED_DIM), ('num_experts', NUM_EXPERTS),
                   ('batch_size', BATCH_SIZE), ('epochs', EPOCHS), ('lr', LR)]:
    cfg_text = _patch(cfg_text, field, val)
with open('config.py', 'w') as f:
    f.write(cfg_text)

if 'config' in sys.modules:
    import config as _c; importlib.reload(_c); from config import CFG
else:
    from config import CFG

print("=" * 55)
print(f"  backbone     : {CFG.backbone}")
print(f"  feature_mode : {CFG.feature_mode}  →  feature_dim={CFG.feature_dim}")
print(f"  feature_dir  : {CFG.feature_dir}")
print(f"  embed_dim    : {CFG.embed_dim}  epochs={CFG.epochs}  lr={CFG.lr}")
print("=" * 55)

## Cell 5 — Load / Kiểm tra Features

Các phương pháp này đều dùng **pre-extracted features** (không cần train backbone từ đầu).
Nếu chưa có features, chạy Cell 5–6 trong `kaggle_train.ipynb` trước.

In [ ]:
import os
from config import CFG

SENTINEL = os.path.join(CFG.feature_dir, 'pathmnist_train_feat.npy')
if os.path.exists(SENTINEL):
    npy = [f for f in os.listdir(CFG.feature_dir) if f.endswith('.npy')]
    print(f'✅ Features sẵn sàng: {len(npy)} files tại {CFG.feature_dir}')
else:
    print(f'❌ Features chưa có tại {CFG.feature_dir}')
    print('   → Chạy kaggle_train.ipynb Cell 5+6 để load/extract features trước.')
    raise FileNotFoundError(f'Missing: {SENTINEL}')

## Cell 6 — GeM (Generalized Mean Pooling)

**Ý tưởng**: Thay vì concat(CLS, PatchMean) cố định, học tham số `p` tối ưu để fuse 2 tokens:
```
f = ((CLS^p + PatchMean^p) / 2)^(1/p)
```
- `p=1` → trung bình đơn giản
- `p→∞` → max pooling
- `p~3` → sweet-spot thường tốt nhất (được học từ data)

~5 phút trên T4.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/gem_baseline.py'], check=True)

## Cell 7 — DOLG (Dynamic Orthogonal Local-Global)

**Ý tưởng**: Tách biệt vai trò của CLS (global) và PatchMean (local):
```
l_orth = PatchMean - proj(PatchMean onto CLS)   # loại bỏ phần trùng với global
fused  = CLS + l_orth                           # chỉ giữ thông tin mới từ local
```
Local features chỉ bổ sung thông tin mà global CHƯA có (orthogonal).

~5 phút trên T4.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/dolg_baseline.py'], check=True)

## Cell 8 — CSQ (Central Similarity Quantization)

**Ý tưởng**: Học binary hash codes (64-bit) cho retrieval:
- Mỗi class có một hash center cố định {-1,+1}^64
- Loss kéo features về hash center của class mình
- Retrieval: cosine similarity trên real-valued codes / Hamming distance trên binary codes

~5 phút trên T4.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'baselines/csq_baseline.py', '--n_bits', '64'], check=True)

## Cell 9 — SuperGlobal Re-ranking (α-QE)

**Ý tưởng**: Post-processing không cần retrain. Áp dụng trên embeddings của model bất kỳ:
```
q' = normalize(q + Σ sim(q, g_i)^α * g_i)   for top-K neighbors g_i
```
Sharpen query bằng cách aggregate top-K kết quả. Không thay đổi gallery.

Chạy trên MoE (nếu đã train) hoặc đổi `--model gem/dolg/csq`.

In [ ]:
import subprocess, sys

# Chạy SuperGlobal trên từng model (uncomment model muốn re-rank)
for model_name in ['moe', 'gem', 'dolg', 'csq']:
    ckpt = f'results/checkpoints/best_{model_name}.pt'
    import os
    if not os.path.exists(ckpt):
        print(f'⏭️  Bỏ qua {model_name} — chưa có checkpoint: {ckpt}')
        continue
    print(f'\n>>> SuperGlobal re-ranking: {model_name}')
    subprocess.run([
        sys.executable, 'baselines/superglobal.py',
        '--model', model_name,
        '--alpha', '3.0',
        '--top_k', '10',
    ], check=True)

## Cell 10 — So sánh kết quả

In [ ]:
import pandas as pd, os

methods = {
    'gem':  'results/history_gem.csv',
    'dolg': 'results/history_dolg.csv',
    'csq':  'results/history_csq.csv',
    'moe':  'results/history_moe.csv',
    'linear': 'results/history_linear.csv',
    'mlp':  'results/history_mlp.csv',
}

rows = []
for name, path in methods.items():
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    best_row = df.loc[df['avg_mAP@R'].idxmax()]
    row = {'Method': name.upper(), 'Best mAP@R': best_row['avg_mAP@R']}
    # Per-dataset columns
    from config import CFG
    for ds in CFG.datasets:
        col = f'{ds}.mAP@R'
        if col in df.columns:
            row[ds[:4]] = best_row[col]
    rows.append(row)

if rows:
    df_cmp = pd.DataFrame(rows).sort_values('Best mAP@R', ascending=False)
    print(df_cmp.to_string(index=False))
else:
    print('Chưa có kết quả nào. Chạy Cell 6–9 trước.')

## Cell 11 — Save Results

Tất cả file trong `/kaggle/working/` tự động lưu — tải về qua tab **Output**.

In [ ]:
import os
for root, dirs, files in os.walk('results'):
    for f in sorted(files):
        path = os.path.join(root, f)
        print(f'{path}  ({os.path.getsize(path)/1024:.1f} KB)')